In [ ]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier

# ============================================================
# 1. XAC DINH DUONG DAN THU MUC DU AN
# ============================================================

current_dir = Path.cwd().resolve()

if current_dir.name == "experiments":
    project_root = current_dir.parent
elif (current_dir / "src").is_dir():
    project_root = current_dir
else:
    raise FileNotFoundError(
        "Khong tim thay thu muc goc DeTaiCuKy_V2 chua src"
    )

sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Thu muc src:", (project_root / "src").exists())

# ============================================================
# 2. IMPORT CAC MODULE TRONG SRC
# ============================================================

from src.data import load_data
from src.config import RANDOM_STATE, CV_DIR
from src.evaluation import cv_evaluate
from src.pipelines import get_xgb
from src.logger import ExperimentLogger

# ============================================================
# 3. TAI DU LIEU
# ============================================================

X_train, _, y_train, _, labels = load_data()

lg = ExperimentLogger("tuan")

# ============================================================
# 4. KHAI BAO CAC MO HINH CAN SO SANH
# ============================================================

ung_vien = {
    "hgb_N1_balanced_rep3": HistGradientBoostingClassifier(
        class_weight="balanced",
        early_stopping=False,
        random_state=RANDOM_STATE
    ),

    "xgb_default_rep3": get_xgb()["xgb_default"],
}

# ============================================================
# 5. CHAY CROSS-VALIDATION LAP 3 LAN
# ============================================================

for ten, model in ung_vien.items():

    cv_df = cv_evaluate(
        model,
        X_train,
        y_train,
        ten,
        n_repeats=3
    )

    lg.log(
        model,
        cv_df,
        ten,
        strategy="CV lap 3x5",
        notes="kiem tra on dinh cho ung vien cuoi"
    )

    print(
        ten,
        "macro_f1", round(cv_df["macro_f1"].mean(), 4),
        "std", round(cv_df["macro_f1"].std(), 4),
        "recall_c11", round(cv_df["recall_c11"].mean(), 4)
    )

# ============================================================
# 6. DOC KET QUA CROSS-VALIDATION
# ============================================================

a = pd.read_csv(
    CV_DIR / "hgb_N1_balanced_rep3.csv"
)["macro_f1"].to_numpy()

b = pd.read_csv(
    CV_DIR / "xgb_default_rep3.csv"
)["macro_f1"].to_numpy()

# ============================================================
# 7. KIEM DINH T HIEU CHINH
# ============================================================

d = b - a
k = len(d)

t_hc = d.mean() / np.sqrt(
    (1 / k + 0.25) * d.var(ddof=1)
)

p_value = 2 * stats.t.sf(
    abs(t_hc),
    df=k - 1
)

print("xgb tru hgb_N1:", round(d.mean(), 4))
print("t hieu chinh:", round(t_hc, 2))
print("p:", round(p_value, 3))
